In [1]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append("../scripts")

from static_encoding import process_embeddings

In [4]:
EMBEDDING_PATH = r"C:\Users\mayat\OneDrive\Desktop\lab\Language-Project-main\data\processed\podcast_trilingual_embeddings.csv"

BIDS_ROOT = r"C:\Users\mayat\ds005574-download/"

In [6]:
import numpy as np
import pandas as pd
import os

# Paths
EMBEDDING_PATH = r"C:\Users\mayat\OneDrive\Desktop\lab\Language-Project-main\data\processed\podcast_trilingual_embeddings.csv"

BIDS_ROOT = r"C:\Users\mayat\ds005574-download/"

OUTPUT_DIR = r"C:\Users\mayat\OneDrive\Desktop\lab\Language-Project-main\data\processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Subjects to run
subjects = ["01", "02", "03", "04", "05", "06", "07", "08", "09"]

# Conditions to test
conditions = {
    "English static": "en",
    "Hebrew static": "he",
    "Arabic static": "ar",
    "English + Hebrew": "en+he",
    "English + Arabic": "en+ar",
    "English + Hebrew + Arabic": "all"
}

all_results = []
all_scores = {}

for subj in subjects:
    print("\n" + "=" * 70)
    print(f"Running subject {subj}")
    print("=" * 70)

    for condition_name, language_mode in conditions.items():
        print(f"\nSubject {subj} | Condition: {condition_name}")

        try:
            raw, scores = process_embeddings(
                subj=subj,
                bids_root=BIDS_ROOT,
                embedding_filename=EMBEDDING_PATH,
                language_mode=language_mode,
                freq=32,
                tmin=-1.0,
                tmax=1.0,
                use_PCA=False
            )

            mean_corr = np.nanmean(scores)
            median_corr = np.nanmedian(scores)
            max_corr = np.nanmax(scores)

            print(f"Mean correlation: {mean_corr:.6f}")

            all_results.append({
                "subject": subj,
                "condition": condition_name,
                "language_mode": language_mode,
                "mean_corr": mean_corr,
                "median_corr": median_corr,
                "max_corr": max_corr,
                "scores_shape": str(scores.shape)
            })

            # Save full scores in memory for later
            all_scores[(subj, condition_name)] = scores

            # Save full score array per subject-condition
            score_filename = f"static_scores_sub{subj}_{language_mode.replace('+', '_plus_')}.npy"
            np.save(os.path.join(OUTPUT_DIR, score_filename), scores)

        except Exception as e:
            print(f"FAILED subject {subj}, condition {condition_name}: {e}")

            all_results.append({
                "subject": subj,
                "condition": condition_name,
                "language_mode": language_mode,
                "mean_corr": np.nan,
                "median_corr": np.nan,
                "max_corr": np.nan,
                "scores_shape": "FAILED",
                "error": str(e)
            })

# Create summary table
results_all_subjects = pd.DataFrame(all_results)

# Save summary CSV
summary_path = os.path.join(OUTPUT_DIR, "static_encoding_results_all_subjects.csv")
results_all_subjects.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("\nSaved summary to:")
print(summary_path)

results_all_subjects


Running subject 01

Subject 01 | Condition: English static
Himalaya backend: Using Torch (CPU)
Mean correlation: 0.004817

Subject 01 | Condition: Hebrew static
Himalaya backend: Using Torch (CPU)
Mean correlation: 0.002547

Subject 01 | Condition: Arabic static
Himalaya backend: Using Torch (CPU)
Mean correlation: 0.001911

Subject 01 | Condition: English + Hebrew
Himalaya backend: Using Torch (CPU)
Mean correlation: 0.004426

Subject 01 | Condition: English + Arabic
Himalaya backend: Using Torch (CPU)
Mean correlation: 0.004271

Subject 01 | Condition: English + Hebrew + Arabic
Himalaya backend: Using Torch (CPU)
Mean correlation: 0.004133

Running subject 02

Subject 02 | Condition: English static
Himalaya backend: Using Torch (CPU)
Mean correlation: 0.011818

Subject 02 | Condition: Hebrew static
Himalaya backend: Using Torch (CPU)
Mean correlation: 0.010544

Subject 02 | Condition: Arabic static
Himalaya backend: Using Torch (CPU)
Mean correlation: 0.006507

Subject 02 | Conditio

,subject,condition,language_mode,mean_corr,median_corr,max_corr,scores_shape
0,01,English static,en,0.004817,0.004176,0.169920,"(2, 99, 64)"
1,01,Hebrew static,he,0.002547,0.002020,0.191546,"(2, 99, 64)"
2,01,Arabic static,ar,0.001911,0.001702,0.172736,"(2, 99, 64)"
3,01,English + Hebrew,en+he,0.004426,0.003582,0.201920,"(2, 99, 64)"
4,01,English + Arabic,en+ar,0.004271,0.003883,0.192484,"(2, 99, 64)"
5,01,English + Hebrew + Arabic,all,0.004133,0.003521,0.199149,"(2, 99, 64)"
6,02,English static,en,0.011818,0.011621,0.190125,"(2, 90, 64)"
7,02,Hebrew static,he,0.010544,0.009610,0.174143,"(2, 90, 64)"
8,02,Arabic static,ar,0.006507,0.006677,0.188539,"(2, 90, 64)"
9,02,English + Hebrew,en+he,0.012990,0.012145,0.192232,"(2, 90, 64)"


In [7]:
condition_summary = (
    results_all_subjects
    .groupby("condition", as_index=False)
    .agg(
        mean_across_subjects=("mean_corr", "mean"),
        std_across_subjects=("mean_corr", "std"),
        n_subjects=("mean_corr", "count")
    )
    .sort_values("mean_across_subjects", ascending=False)
)

condition_summary

,condition,mean_across_subjects,std_across_subjects,n_subjects
2,English + Hebrew,0.003999,0.005765,9
4,English static,0.003740,0.004356,9
5,Hebrew static,0.003044,0.006166,9
3,English + Hebrew + Arabic,0.002824,0.007196,9
1,English + Arabic,0.002207,0.006570,9
0,Arabic static,-0.000223,0.007818,9


In [8]:
condition_summary.to_csv(
    os.path.join(OUTPUT_DIR, "static_encoding_condition_summary.csv"),
    index=False,
    encoding="utf-8-sig"
)